# Primer pronóstico oficial — Pipeline de publicación diaria (Mundial 2026)

Este notebook es una **interfaz** sobre el pipeline oficial
`cdd_mundial.live` (D-07). No define lógica de producción: dispara el
comando oficial, inspecciona las rutas de los artefactos publicados y
muestra el reporte. La autoridad vive en `src/cdd_mundial/live/` y en
`python -m cdd_mundial.live`; aquí solo orquestamos e inspeccionamos.

Cadena oficial (orden inviolable):
`materialize → select_model → simulate → publish` + render del reporte.

## Importaciones e interfaz (What and why · qué y por qué)

Importamos únicamente las funciones públicas del pipeline oficial
(`verify_official`, `run_official`) y el renderer de snapshots. No
reimplementamos materialización, simulación ni calibración: son
responsabilidad de `src/`. `verify_official` permite un *dry-run* sin
escribir; `run_official` ejecuta la publicación append-only completa.

In [1]:
import os
from pathlib import Path
import json

# El kernel arranca en notebooks/; nos paramos en la raiz del repo para que
# las rutas relativas (results, fixture, snapshots) resuelvan igual que en
# `python -m cdd_mundial.live`. Idempotente: no hace nada si ya estamos en la raiz.
if not Path('pyproject.toml').exists():
    os.chdir(Path.cwd().parent)

from cdd_mundial.live.pipeline import run_official, verify_official
from cdd_mundial.live.report import render_snapshot_report

RESULTS = Path('data/external/results_2026.csv')
FIXTURE = Path('data/external/fixture_2026.csv')
MANUAL_ODDS = Path('data/external/odds_2026_template.csv')
SNAPSHOTS_ROOT = Path('reports/snapshots')
SEED = 20260613
print('Interfaz lista. Pipeline oficial: python -m cdd_mundial.live')


Interfaz lista. Pipeline oficial: python -m cdd_mundial.live


**Interpretation.** Quedan disponibles las dos entradas del pipeline
oficial. Cualquier corrida desde el notebook es idéntica a la del
comando de terminal porque ambas llaman al mismo código de `src/`.

## Dry-run: verificar prerequisitos sin publicar (What and why · qué y por qué)

Antes de publicar, `verify_official` valida los resultados, materializa
el artefacto live-training inmutable, resuelve el fingerprint y la
decisión reuse/refit, y reporta el orden previsto — **sin** escribir un
snapshot. Es el preflight reproducible del operador.

In [2]:
preview = verify_official(
    results_path=RESULTS,
    fixture_path=FIXTURE,
    snapshots_root=SNAPSHOTS_ROOT,
    allow_dirty=True,  # solo para inspección; el run real exige worktree limpio
)
print(json.dumps(preview, indent=2, sort_keys=True))

{
  "as_of_date": "2026-06-14",
  "dirty": true,
  "input_fingerprint": "d0bfa14dbde543f679c671601c2e0aeb56d73c0d21721cff6edb7ec4084bd460",
  "live_training_path": "data/processed/live/live_training_2026-06-14.parquet",
  "live_training_sha256": "56aaf68afc162d3e106346dbae170ea2643d021ebf5931905352388ea4421dd8",
  "model_path": "data/processed/models/dc_params_2026-06-14.json",
  "model_version": "baseline-v1-2026-06-14-d0bfa14",
  "order": [
    "materialize",
    "select_model",
    "simulate",
    "publish"
  ],
  "published": false,
  "reused_model": false
}


**Interpretation.** El dry-run confirma el orden
`[materialize, select_model, simulate, publish]`, el SHA-256 del
artefacto live-training, el `input_fingerprint` y el `model_version`
datado. `published=false`: no se escribió nada.

## Publicación oficial append-only (What and why · qué y por qué)

La publicación real exige **worktree limpio** (D-11) y se commitea
*antes* del `kickoff_boundary_utc`. Desde el notebook se invoca igual que
el comando de terminal. Aquí lo dejamos **comentado** para no publicar un
snapshot por accidente al re-ejecutar el notebook: el snapshot oficial se
genera con el comando del runbook. Descoméntalo solo para una corrida
real con el worktree limpio.

In [3]:
# summary = run_official(
#     results_path=RESULTS,
#     fixture_path=FIXTURE,
#     snapshots_root=SNAPSHOTS_ROOT,
#     manual_odds_path=MANUAL_ODDS,
#     seed=SEED,
# )
# print(json.dumps(summary, indent=2, sort_keys=True))

# Modo interfaz: inspeccionamos el último snapshot ya publicado.
published = sorted(
    (d for d in SNAPSHOTS_ROOT.iterdir() if d.is_dir() and not d.name.startswith('.')),
    key=lambda d: d.name,
) if SNAPSHOTS_ROOT.exists() else []
latest = published[-1] if published else None
print('Snapshots publicados:', len(published))
print('Último snapshot:', latest.name if latest else 'ninguno')

Snapshots publicados: 2
Último snapshot: 2026-06-13T22-07-38Z_baseline-v1-2026-06-13-50d98ab


**Interpretation.** En modo interfaz no republicamos: localizamos el
snapshot oficial más reciente bajo `reports/snapshots/`. Si quieres
generar uno nuevo, descomenta `run_official(...)` con el worktree limpio.

## Inspección del bundle publicado (What and why · qué y por qué)

Un snapshot oficial es un directorio inmutable. Leemos su `metadata.json`
para auditar la procedencia: live-training, modelo, semilla, boundary
pre-kickoff, commit, estado dirty y checksums.

In [4]:
assert latest is not None, 'No hay snapshot publicado; corre el comando del runbook.'
meta = json.loads((latest / 'metadata.json').read_text(encoding='utf-8'))
for key in [
    'generated_at_utc', 'kickoff_boundary_utc', 'git_commit', 'git_dirty',
    'seed', 'model_version',
]:
    print(f'{key}: {meta.get(key)}')
print('live_training:', meta['live_training_provenance']['artifact_path'])
print('fingerprint:', meta['model_provenance']['input_fingerprint'][:12], '...')
print('publication_row_ids:', len(meta['publication_row_ids']))
print('artefactos:', sorted(meta['checksums']))

generated_at_utc: 2026-06-13T22:07:38Z
kickoff_boundary_utc: 2026-06-14T01:00:00Z
git_commit: f79070114ed6ea548559ec6ca7b15607843b11f0
git_dirty: False
seed: 20260613
model_version: baseline-v1-2026-06-13-50d98ab
live_training: data/processed/live/live_training_2026-06-13.parquet
fingerprint: 50d98ab2142b ...
publication_row_ids: 3
artefactos: ['frozen_benchmark.parquet', 'group_positions.parquet', 'report_inputs/calibration_publication_slice.parquet', 'team_probabilities.parquet', 'upcoming_match_predictions.parquet']


**Interpretation.** La metadata permite reconstruir exactamente la
corrida oficial: el artefacto live-training con su SHA-256, el modelo con
su fingerprint, la semilla, y el boundary pre-kickoff que la publicación
respetó. Los checksums sellan cada tabla del bundle.

## Tablas y reporte del snapshot (What and why · qué y por qué)

Las probabilidades por equipo y los pronósticos del próximo bloque viven
como parquet congelados. El `report.html` se renderiza desde esos
artefactos (nunca re-simula). Re-renderizar el mismo snapshot es
determinista.

In [5]:
import pandas as pd
team_probs = pd.read_parquet(latest / 'team_probabilities.parquet')
top = team_probs.sort_values('p_champion', ascending=False).head(8)
print(top[['team_id', 'p_champion']].to_string(index=False))
print()
print('Reporte HTML:', (latest / 'report.html').exists())
print('Assets:', [p.name for p in (latest / 'assets').glob('*.png')])

  team_id  p_champion
argentina      0.2138
    spain      0.1430
  england      0.1012
   brazil      0.0958
   france      0.0646
 portugal      0.0643
 colombia      0.0429
  belgium      0.0414

Reporte HTML: True
Assets: ['highlight_champion.png', 'tournament_champion.png']


**Interpretation.** El top de favoritos sale del snapshot congelado, no
de una nueva simulación. El `report.html` + `assets/` son el artefacto
publicable diario. Para regenerarlo sin mutar el bundle, usa
`render_snapshot_report(latest, snapshots_root=SNAPSHOTS_ROOT,
ledger_path=..., output_dir=<sibling>)`.

## Cierre

Este notebook disparó e inspeccionó la corrida oficial **sin** duplicar
lógica de negocio. El flujo de operación diaria está en
`docs/live_pipeline_runbook.md`; la autoridad reproducible en
`python -m cdd_mundial.live`.